In [0]:
%run ../Notebooks/00_Configuration

Configuration Loaded Successfully


In [0]:
%run ../framework/01_Utility_Functions

Configuration Loaded Successfully


In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("pipeline_run_id", "")

from pyspark.sql.functions import col, upper, lit, current_timestamp
from pyspark.sql.utils import AnalysisException
from delta.tables import DeltaTable
from functools import reduce

pipeline_run_id = dbutils.widgets.get("pipeline_run_id").strip()

print("=" * 80)
print("ENTERPRISE GENERIC SCD2 LOADER V4")
print("=" * 80)
print("Pipeline Run :", pipeline_run_id)

ENTERPRISE GENERIC SCD2 LOADER V4
Pipeline Run : 


In [0]:
NOTEBOOK_NAME = "Enterprise_Generic_SCD2_Loader_v4"

def print_header(title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

def print_info(name, value):
    print(f"{name:<30}: {value}")

def print_success(message):
    print(f"SUCCESS : {message}")

def print_warning(message):
    print(f"WARNING : {message}")

def print_error(message):
    print(f"ERROR   : {message}")

In [0]:
print_header("READING ALL SCD2-ENABLED METADATA")

# Fixed: original notebook read spark.table("insurance_metadata.bronze_config") with no
# catalog/schema prefix, which only worked by accident if that was the default catalog/schema
# in the session. Uses the same fully-qualified pattern as Bronze/Silver/Gold now.
metadata_rows = (
    spark.table(f"{catalog_name}.{metadata_schema}.bronze_config")
         .filter(upper(col("active")) == "Y")
         .filter(upper(col("scd_enabled")) == "Y")
         .orderBy("table_name")
         .collect()
)

if len(metadata_rows) == 0:
    raise Exception("No active, SCD2-enabled tables found in bronze_config.")

print_success(f"{len(metadata_rows)} active SCD2-enabled table(s) found.")
for r in metadata_rows:
    print_info("Active Table", r["table_name"])


READING ALL SCD2-ENABLED METADATA
SUCCESS : 3 active SCD2-enabled table(s) found.
Active Table                  : Agent
Active Table                  : Branch
Active Table                  : Customer


In [0]:
print_header("PROCESSING ALL SCD2-ENABLED TABLES")

results_summary = []

for config in metadata_rows:

    table_name = config["table_name"]

    print_header(f"PROCESSING {table_name}")

    try:
        # ---------------- Metadata ----------------
        primary_key   = config["primary_key"]
        gold_table    = config["gold_table"]
        history_table = config["history_table"]
        raw_compare   = config["compare_columns"]

        if primary_key is None or str(primary_key).strip() == "":
            raise Exception("Mandatory metadata field 'primary_key' is missing.")
        if gold_table is None or str(gold_table).strip() == "":
            raise Exception("Mandatory metadata field 'gold_table' is missing.")
        if history_table is None or str(history_table).strip() == "":
            raise Exception("Mandatory metadata field 'history_table' is missing.")
        if raw_compare is None or str(raw_compare).strip() == "":
            raise Exception("Mandatory metadata field 'compare_columns' is missing.")

        compare_columns = [c.strip() for c in raw_compare.split(",") if c.strip() != ""]

        gold_table_name = f"{catalog_name}.{gold_schema}.{gold_table}"
        history_table_name = f"{catalog_name}.{gold_schema}.{history_table}"

        print_info("Primary Key", primary_key)
        print_info("Gold Table", gold_table_name)
        print_info("History Table", history_table_name)
        print_info("Compare Columns", compare_columns)

        # ---------------- Read Gold ----------------
        if not spark.catalog.tableExists(gold_table_name):
            raise Exception(f"Gold table does not exist yet : {gold_table_name}")

        gold_df = spark.table(gold_table_name)
        gold_rows = gold_df.count()
        print_info("Gold Rows", gold_rows)

        missing_cols = [c for c in compare_columns if c not in gold_df.columns]
        if missing_cols:
            raise Exception(f"compare_columns not found in Gold table: {missing_cols}")

        # ---------------- Create history table on first run ----------------
        if not spark.catalog.tableExists(history_table_name):

            (
                gold_df
                .withColumn("effective_start_date", current_timestamp())
                .withColumn("effective_end_date", lit(None).cast("timestamp"))
                .withColumn("is_current", lit(True))
                .write
                .format("delta")
                # .option("overwriteSchema","true")
                .mode("overwrite")
                .saveAsTable(history_table_name)
            )

            print_success("History table created.")

        else:
            print_info("History table", "already exists")

        history_df = spark.table(history_table_name)

        # ---------------- Compare current history vs Gold ----------------
        current_history_df = history_df.filter(col("is_current") == True)
        current_history_rows = current_history_df.count()
        print_info("Current History Rows", current_history_rows)

        history_columns = []
        for column in current_history_df.columns:
            if column == primary_key:
                history_columns.append(col(column))
            else:
                history_columns.append(col(column).alias(f"old_{column}"))

        history_renamed_df = current_history_df.select(*history_columns)

        comparison_df = (
            gold_df.alias("new")
            .join(history_renamed_df.alias("old"), on=primary_key, how="left")
        )

        # ---------------- New records ----------------
        new_records = comparison_df.filter(col("old.old_effective_start_date").isNull())
        new_record_count = new_records.count()
        print_info("New Records", new_record_count)

        # ---------------- Changed records ----------------
        comparison_conditions = [
            ~col(f"new.{c}").eqNullSafe(col(f"old.old_{c}"))
            for c in compare_columns
        ]
        change_condition = reduce(lambda x, y: x | y, comparison_conditions)

        changed_records = comparison_df.filter(
            change_condition & col("old.old_effective_start_date").isNotNull()
        )
        changed_record_count = changed_records.count()
        print_info("Changed Records", changed_record_count)

        # ---------------- Expire changed versions ----------------
        if changed_record_count > 0:

            history_delta = DeltaTable.forName(spark, history_table_name)

            changed_keys = changed_records.select(primary_key).distinct()

            (
                history_delta.alias("hist")
                .merge(
                    changed_keys.alias("chg"),
                    f"hist.{primary_key} = chg.{primary_key} AND hist.is_current = true"
                )
                .whenMatchedUpdate(
                    set={
                        "is_current": "false",
                        "effective_end_date": "current_timestamp()"
                    }
                )
                .execute()
            )

            print_success("Existing versions expired.")
        else:
            print_info("Existing versions expired", "none required")

        # ---------------- Insert new + changed as new current versions ----------------
        new_insert_df = new_records.select("new.*")
        changed_insert_df = changed_records.select("new.*")

        final_insert_df = (
            new_insert_df
            .unionByName(changed_insert_df)
            .withColumn("effective_start_date", current_timestamp())
            .withColumn("effective_end_date", lit(None).cast("timestamp"))
            .withColumn("is_current", lit(True))
        )

        insert_count = final_insert_df.count()
        print_info("Rows To Insert", insert_count)

        if insert_count > 0:
            (
                final_insert_df.write
                .format("delta")
                .mode("append")
                .saveAsTable(history_table_name)
            )
            print_success(f"Inserted {insert_count} new SCD2 record(s).")
        else:
            print_info("Inserted", 0)

        # ---------------- Audit ----------------
        rows_written = new_record_count + changed_record_count

        write_audit(
            pipeline_name=NOTEBOOK_NAME,
            table_name=history_table,
            load_type="SCD2",
            rows_read=gold_rows,
            rows_written=rows_written,
            status="SUCCESS",
            error_message="",
            pipeline_run_id=pipeline_run_id
        )
        print_success("Audit Written")

        final_history_df = spark.table(history_table_name)

        results_summary.append({
            "table_name": table_name,
            "status": "SUCCESS",
            "gold_rows": gold_rows,
            "new_records": new_record_count,
            "changed_records": changed_record_count,
            "history_rows": final_history_df.count(),
            "error": ""
        })

    except Exception as ex:

        print_error(str(ex))

        write_audit(
            pipeline_name=NOTEBOOK_NAME,
            table_name=table_name,
            load_type="SCD2",
            rows_read=0,
            rows_written=0,
            status="FAILED",
            error_message=str(ex),
            pipeline_run_id=pipeline_run_id
        )

        results_summary.append({
            "table_name": table_name,
            "status": "FAILED",
            "gold_rows": 0,
            "new_records": 0,
            "changed_records": 0,
            "history_rows": 0,
            "error": str(ex)
        })

        continue


PROCESSING ALL SCD2-ENABLED TABLES

PROCESSING Agent
Primary Key                   : agent_id
Gold Table                    : dbw_insurance.insurance_gold.dim_agent
History Table                 : dbw_insurance.insurance_gold.dim_agent_history
Compare Columns               : ['agent_name', 'agent_email', 'agent_phone']
Gold Rows                     : 1000
SUCCESS : History table created.
Current History Rows          : 1000
New Records                   : 0
Changed Records               : 0
Existing versions expired     : none required
Rows To Insert                : 0
Inserted                      : 0
SUCCESS : Audit Written

PROCESSING Branch
Primary Key                   : branch_id
Gold Table                    : dbw_insurance.insurance_gold.dim_branch
History Table                 : dbw_insurance.insurance_gold.dim_branch_history
Compare Columns               : ['branch_country', 'branch_city']
Gold Rows                     : 1000
SUCCESS : History table created.
Current History 

In [0]:
print("\n")
print("=" * 90)
print("ENTERPRISE GENERIC SCD2 LOADER V4 COMPLETED")
print("=" * 90)

for r in results_summary:
    line = (
        f"{r['table_name']:<15} : {r['status']:<8} "
        f"Gold={r['gold_rows']:<6} New={r['new_records']:<5} "
        f"Changed={r['changed_records']:<5} History={r['history_rows']:<6}"
    )
    if r["status"] == "FAILED":
        line += f"  ERROR: {r['error']}"
    print(line)

success_count = len([r for r in results_summary if r["status"] == "SUCCESS"])
failed_count = len([r for r in results_summary if r["status"] == "FAILED"])

print("=" * 90)
print_success(f"Processed {len(results_summary)} tables. Success={success_count}  Failed={failed_count}")
print("=" * 90)



ENTERPRISE GENERIC SCD2 LOADER V4 COMPLETED
Agent           : SUCCESS  Gold=1000   New=0     Changed=0     History=1000  
Branch          : SUCCESS  Gold=1000   New=0     Changed=0     History=1000  
Customer        : SUCCESS  Gold=1958   New=2     Changed=0     History=1960  
SUCCESS : Processed 3 tables. Success=3  Failed=0


In [0]:
# %sql
# DROP TABLE IF EXISTS insurance_gold.dim_customer_history;

# DROP TABLE IF EXISTS insurance_gold.dim_agent_history;

# DROP TABLE IF EXISTS insurance_gold.dim_branch_history;

In [0]:
# history_df = spark.table(f"{catalog_name}.{gold_schema}.dim_customer_history")

# history_df.groupBy("customer_id").count().filter("count > 1").show(truncate=False)

In [0]:
# gold_df = spark.table(f"{catalog_name}.{gold_schema}.dim_customer")

# gold_df.groupBy("customer_id") \
#        .count() \
#        .filter("count > 1") \
#        .orderBy("customer_id") \
#        .show(100, truncate=False)

In [0]:
# %sql
# SELECT *
# FROM insurance_silver.customer
# WHERE customer_id = 1001;

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:440)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:465)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:750)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperation$1(UsageLogging.scala:510)
	at com.databricks.logging.UsageLogging.executeThunkAndCaptureResultTags$1(UsageLogging.scala:616)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperationWithResultTags$4(UsageLogging.scala:643)
	at com.databricks.logging.AttributionContextTracing.$anonfun$withAttributionContext$1(AttributionContextTracing.scala:80)
	at com.databricks.logging.AttributionContext$.$anonfun$withValue$1(AttributionContext.scala:348)
	at scala.util.DynamicVariable.withValue(DynamicVariable.scala:59)
	at com.databricks.logging.AttributionContext$.withValue(Attr

In [0]:
%sql
-- SELECT
-- customer_id,
-- _pipeline_run_id,
-- COUNT(*)
-- FROM insurance_silver.customer
-- GROUP BY customer_id,_pipeline_run_id
-- ORDER BY customer_id;

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:440)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:465)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:750)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperation$1(UsageLogging.scala:510)
	at com.databricks.logging.UsageLogging.executeThunkAndCaptureResultTags$1(UsageLogging.scala:616)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperationWithResultTags$4(UsageLogging.scala:643)
	at com.databricks.logging.AttributionContextTracing.$anonfun$withAttributionContext$1(AttributionContextTracing.scala:80)
	at com.databricks.logging.AttributionContext$.$anonfun$withValue$1(AttributionContext.scala:348)
	at scala.util.DynamicVariable.withValue(DynamicVariable.scala:59)
	at com.databricks.logging.AttributionContext$.withValue(Attr

In [0]:
# %sql
# SELECT
# customer_id,
# _ingestion_timestamp
# FROM insurance_silver.customer
# WHERE customer_id=1001;

In [0]:
# %sql
# DESCRIBE HISTORY insurance_silver.customer;

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:132)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:132)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:190)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:721)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:441)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:441)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
# %sql
# SELECT
# MAX(_silver_processed_timestamp),
# MIN(_silver_processed_timestamp)
# FROM insurance_silver.customer;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5587530983445909>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'SELECT\nMAX(_silver_processed_timestamp),\nMIN(_silver_processed_timestamp)\nFROM insurance_silver.customer;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:183, in SqlMagic.sql(self, line, cell)
    

In [0]:
# %sql
# DROP TABLE insurance_silver.customer;

In [0]:
# %sql
# DROP TABLE insurance_gold.dim_customer_history;